In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Section 01
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    MLX Inference<br>
    <span style="color:#63b3ed;">Multi-Model Comparison on Your Mac</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    Apple Silicon's unified memory changes everything. Run any combination of
    local models side-by-side to see the quality/speed tradeoff viscerally.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🍎 Apple Silicon</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🧠 MLX</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🔌 OpenAI API</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🏆 N Models</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~20 min</span>
  </div>
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💻 Step 1: Hardware Check</h2>
</div>

First — what are we working with?

**Why hardware matters for LLMs.** Running a large language model locally is almost entirely a memory bandwidth problem. Every token generated requires loading billions of weight values from storage into compute units — the faster that transfer happens, the higher your token-per-second rate. This is fundamentally different from training, which is compute-bound.

**Unified memory vs discrete GPU.** On a conventional PC, the CPU and GPU have separate memory pools connected by PCIe — a bus that tops out at roughly 32–64 GB/s. Every time a model layer needs to move data between CPU RAM and GPU VRAM, it crosses that bottleneck. Apple Silicon uses a **unified memory architecture (UMA)**: the CPU, GPU, and Neural Engine all read from the same physical DRAM at the full SoC fabric speed — up to ~546 GB/s on M4 Max. No copying. No PCIe penalty.

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> Think of a discrete GPU like having a separate SRAM chip connected to your MCU over SPI — fast enough for small transfers, but the bus becomes the bottleneck for bulk data. Apple Silicon is more like placing that SRAM directly on the same die as your processor, sharing the same high-speed internal bus. The memory pool is unified; arbitration is handled in hardware.
</div>

**Memory bandwidth as the speed limit.** The formula is simple: `max tok/s ≈ bandwidth_GB_s / (active_params_B × bytes_per_param)`. With a 4-bit quantized 7B model (≈3.5 GB) and 400 GB/s bandwidth, the theoretical ceiling is over 100 tok/s. You'll see numbers close to that below. Think of it like flash read speed limiting how fast your bootloader can copy firmware into SRAM — the CPU isn't the bottleneck, the bus is.

In [2]:
!python3 ../../scripts/setup_check.py


┌────────────────────────────────────────────────────────────┐
│                    Hardware Setup Check                    │
└────────────────────────────────────────────────────────────┘

  ▸ Operating System
  ────────────────────────────────────────────────────────
    OS                           Darwin 25.3.0
    Architecture                 arm64
    Python                       3.14.3

  ▸ Chip / Accelerator
  ────────────────────────────────────────────────────────
    Chip                         Apple M4 Max

  ▸ Memory (RAM)
  ────────────────────────────────────────────────────────
    Total RAM                    128.0 GiB
    Available RAM                54.0 GiB

  ▸ MLX Framework
  ────────────────────────────────────────────────────────
    ✓  mlx.core importable                       version=unknown  device=Device(gpu, 0)

  ▸ Model Recommendations
  ────────────────────────────────────────────────────────
    Detected tier: 128 GB+

    Model                       

In [ ]:
# Quick import check — fail fast if wrong kernel
try:
    import openai, psutil
except ImportError as e:
    raise ImportError(
        f"{e}. Select the 'Python 3 (Homebrew)' kernel in Jupyter: "
        "Kernel → Change Kernel → Python 3 (Homebrew)"
    ) from None

import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../scripts").resolve()))
import notebook_helpers

# ── Discover MLX servers ─────────────────────────────────────
# Scans ports 8800-8809 automatically — run as many models as you want.
# Use models WITHOUT vision tower (mlx-lm can't load those):
#   ✓ RepublicOfKorokke/Qwen3.5-*-mlx-lm-*  (text-only, works)
#   ✗ mlx-community/Qwen3.5-*-MLX-*          (has vision tower, crashes)

MODELS, clients = notebook_helpers.discover_servers()

if not MODELS:
    raise RuntimeError(
        "No MLX servers found! Start at least one:\n"
        "  mlx_lm.server --model RepublicOfKorokke/Qwen3.5-2B-mlx-lm-nvfp4 --port 8800"
    )

# ── Warm up with live-updating status table ──────────────────
notebook_helpers.warmup_models(MODELS, clients)
notebook_helpers.init(MODELS, clients)

In [4]:
# Import shared helper functions
from notebook_helpers import strip_think, _md, _render_cards, compare_models, show_metrics_table
print("Helpers loaded.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 Step 2: First Inference — Multi-Model Comparison</h2>
</div>

**What inference means.** At inference time the model weights are *frozen* — no learning is happening. You send in a prompt (a sequence of tokens), it flows through the network as a forward pass, and the model outputs a probability distribution over its vocabulary. We sample from that distribution to pick the next token, append it to the sequence, and repeat. That's it. No backpropagation, no gradient updates.

**OpenAI-compatible API.** MLX serves models over an OpenAI-compatible HTTP API. That means any code you write here — using the `openai` Python client — works identically against GPT-4o, Claude, Gemini, or any other compatible endpoint. You swap the `base_url` and `model` string; everything else stays the same. This portability is why the OpenAI API format became the industry standard.

**Autoregressive generation.** The model generates text one token at a time, left to right. Each new token is conditioned on *all previous tokens* — that's the "auto" in autoregressive. The 2B model generates tokens faster than the 122B model not because it's smarter but because it has far fewer weight values to load from memory per step.

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> Autoregressive generation is like bit-banging a serial protocol in software — you produce one bit (token) at a time, each dependent on the state accumulated so far, with no ability to parallelize across the output sequence. Hardware UART does it in parallel under the hood; similarly, speculative decoding (covered in 01c) tries to draft multiple tokens ahead to break this sequential dependency.
</div>

**Quality scales with size — and so does latency.** Larger models have seen more patterns during training and store more knowledge in their weights. A 122B parameter model will generally produce more accurate, nuanced responses than a 2B model. But it also takes 4–8× longer per token. Watch the responses below and notice both the quality difference *and* the speed difference.

In [5]:
results = compare_models(
    "Explain what you are in exactly 3 sentences, as if you were describing yourself to an electrical engineer who has never heard of an LLM.",
)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📊 Step 3: Measure Performance</h2>
</div>

**tok/s is THE metric.** Tokens per second measures how fast the model generates output during the decode phase. It directly determines the user experience:

| tok/s | Feel |
|-------|------|
| 30+ | Real-time — feels instant |
| 10–30 | Readable — like watching someone type fast |
| 5–10 | Tolerable — noticeable lag |
| < 5 | Painful — hard to use interactively |

**Memory bandwidth is the bottleneck — not compute.** During decode, the model generates one token per forward pass. Each forward pass loads the model weights once from memory. With billions of parameters, that's gigabytes of data streamed per second — far more data movement than compute cycles. Your GPU cores sit mostly idle waiting for the memory bus.

**The bandwidth formula.** A useful rule of thumb:

```
max_tok/s ≈ memory_bandwidth_GB_s / (active_params_billions × bytes_per_param)
```

For a 4-bit quantized 35B MoE model where only 3B parameters are *active* per token: `400 GB/s / (3B × 0.5 bytes) = ~267 tok/s` theoretical max. Real numbers are lower due to overhead, but you can see why MoE models punch above their weight in speed.

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> This is exactly like a flash lookup-table operation on an embedded system — the compute (XOR, add) is trivial, but your throughput is completely limited by how fast you can stream bytes off the flash bus. Wider bus = more tok/s. This is why Apple's high-bandwidth unified memory is such a meaningful advantage for inference.
</div>

In [6]:
# Measure tok/s across all models with a fun prompt
import psutil

results = compare_models(
    "Write a limerick about UART protocol timing violations.",
)

show_metrics_table(results)

ram = psutil.virtual_memory()
print(f"\nSystem RAM used: {(ram.total - ram.available) / 1e9:.1f} GB / {ram.total / 1e9:.0f} GB ({ram.percent}%)")

Model,Tokens,Time,tok/s
2B,1024,4.5s,226.9
35B,37,1.1s,33.8
122B,44,2.5s,18.0



System RAM used: 125.9 GB / 137 GB (91.6%)


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧩 Step 4: Memory Topology</h2>
</div>

Why can Apple Silicon run multiple models simultaneously? **Unified memory.** The CPU, GPU, and Neural Engine share the same RAM — no PCIe bottleneck, no separate VRAM pool.

**Discrete GPU VRAM limits.** A high-end NVIDIA RTX 4090 has 24 GB of GDDR6X VRAM. A 4-bit quantized 34B model needs ~17 GB — leaving barely 7 GB for KV cache and activations. A 70B model simply doesn't fit. If the model overflows VRAM, it spills to system RAM over PCIe (32–64 GB/s) and tok/s craters by 10–20×.

**Unified memory: no VRAM ceiling.** On Apple Silicon, "VRAM" and "RAM" are the same pool. A 128 GB Mac can hold a 65 GB model and still have 60+ GB left for the OS, KV cache, and other models. The GPU accesses all of it at full SoC fabric bandwidth with no copying overhead.

**No PCIe bottleneck.** PCIe 4.0 x16 delivers ~32 GB/s bidirectional. Apple's internal fabric on M4 Max runs at ~546 GB/s — over 17× faster. For memory-bandwidth-limited workloads like LLM inference, this is the entire ballgame.

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> A discrete GPU setup is like an MCU that has separate SRAM (fast, small — the GPU VRAM) and external flash (slow, large — system RAM), connected over SPI. If your data fits in SRAM, you're fast; if it spills to flash, you wait. Apple Silicon is more like a single-die design with one large, fast SRAM that every peripheral accesses directly — no off-chip penalty.
</div>

**Available RAM and swap warnings.** MLX will use macOS swap (on the internal NVMe) if RAM fills up. NVMe swap is ~5–7 GB/s — roughly 80× slower than unified RAM. If you see tok/s suddenly drop to 2–5, you've hit swap. Check `Available RAM` in the cell output and ensure you have headroom before loading another model.

In [ ]:
# Show the memory topology — why Apple Silicon is special for this
import psutil
from IPython.display import HTML, display

ram = psutil.virtual_memory()
used_gb = (ram.total - ram.available) / 1e9

# Build model footprint rows dynamically from selected models
model_rows = ""
for m in MODELS:
    base_label = m["label"].split("-")[0] if "-" in m["label"] else m["label"]
    footprint = notebook_helpers.MODEL_FOOTPRINTS.get(base_label, "unknown")
    model_rows += (
        f"<tr><td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{m['label']}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{footprint}</td></tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Architecture</th>
    <th style="padding:8px 12px; color:white; text-align:left;">GPU Memory</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Access</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;">Discrete GPU</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">24 GB VRAM</td>
        <td style="padding:6px 12px; color:#dc2626; font-weight:bold; border-bottom:1px solid #e5e7eb;">Model must fit here</td></tr>
    <tr><td style="padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;">This Mac</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">{ram.total/1e9:.0f} GB Unified</td>
        <td style="padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;">CPU + GPU + Neural Engine share</td></tr>
  </tbody>
</table>

<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Approx Footprint</th>
  </tr></thead>
  <tbody>
    {model_rows}
  </tbody>
</table>
"""))

print(f"System RAM in use: {used_gb:.1f} GB / {ram.total/1e9:.0f} GB ({ram.percent}%)")
print(f"Available RAM: {ram.available/1e9:.1f} GB")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🌡️ Step 5: Temperature Experiment</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> Temperature controls the randomness of token sampling. At <code>0</code> the model always picks the most likely next token (deterministic). Higher values spread probability across more tokens, making output more creative — or more chaotic.
</div>

**Temperature as a logit scaling factor.** Before sampling, the model produces a vector of raw scores (logits) — one per vocabulary token. Temperature `T` divides every logit before the softmax: `p(token) = softmax(logits / T)`. Lower T sharpens the distribution (winner-take-all); higher T flattens it.

| Temperature | Effect | Use case |
|-------------|--------|----------|
| 0.0 | Argmax — always picks highest-probability token | Classification, structured extraction |
| 0.3–0.5 | Mostly deterministic with slight variation | Q&A, summarization, code |
| 0.7–1.0 | Balanced creativity | General chat, writing assistance |
| > 1.0 | Flattened distribution — chaotic/creative | Brainstorming, poetry, exploration |

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> Temperature is like adjusting the threshold on a comparator. At T=0, any signal above the threshold wins decisively — clean digital output. Raise the threshold voltage (raise T) and more signals compete; the output becomes noisier and less predictable. The circuit still functions, but you've traded determinism for variety.
</div>

**Top-p (nucleus) sampling.** Often used alongside temperature: instead of sampling from the full vocabulary, select the smallest set of tokens whose cumulative probability exceeds `p` (e.g., 0.9), then sample only from those. This caps the "long tail" of improbable tokens that temperature alone can't fully suppress. Most production deployments use both: moderate temperature (0.6–0.8) + top-p (0.9–0.95).

In [9]:
# Temperature = 0.7 across all models — watch the quality spread
results = compare_models(
    "In one sentence, describe how electricity flows through a wire, but make it sound like ancient mythology.",
    temperature=.5,
)

# Try other temperatures yourself — change the value above and re-run!

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🎭 Step 6: System Prompts</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> A system prompt sets the model’s persona and behavior rules. The weights don’t change — you’re just framing the input differently. Think of it as a costume for the model.
</div>

**How system prompts work mechanically.** The system prompt is prepended to the conversation and processed by the same attention mechanism as user messages. It conditions every subsequent token's probability distribution — the model "sees" the system prompt on every decode step because attention spans the entire context. There is no special hardware path or register; it's purely a context-window artifact.

**No weight changes — just probability conditioning.** A system prompt that says "You are a pirate" doesn't modify a single weight. It shifts the probability landscape so that pirate-flavored tokens are consistently more likely given the prior context. Strip the system prompt and the same model responds normally.

**Performance impact.** System prompt tokens are processed during the *prefill* phase (the initial prompt ingestion). This adds to Time-to-First-Token (TTFT) roughly linearly with prompt length. Once prefill completes, the decode speed (tok/s) is unaffected — the system prompt is already cached in the KV store.

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> A system prompt is like a configuration register bank latched at startup. It sets operating mode ("pirate personality", "formal tone", "JSON-only output") before the main loop runs. The core logic (weights) is unchanged — you've just set the mode bits that condition behavior for the entire session.
</div>

**Best practices.** Be specific about format ("respond in bullet points", "answer in under 50 words"). Use positive framing ("always do X" rather than "never do Y" — models handle positive instructions more reliably). Front-load the most important constraints — critical instructions buried deep in long system prompts may be underweighted.

In [10]:
# Pirate persona across all models — entertaining quality spread!
results = compare_models(
    "What is a neural network?",
    system_prompt="You are a pirate who somehow understands machine learning. Arr.",
    temperature=0.8,
)

In [ ]:
# Bonus: 3 personas on the largest available model — streaming in parallel
# ↓ Change this and re-run to see how temperature affects each persona ↓
PERSONA_TEMP = 0.7

import time
import threading
from IPython.display import HTML, display

personas = [
    {"name": "Grumpy Firmware Engineer", "icon": "\U0001f610", "system": "You are a grumpy firmware engineer with 30 years of experience. You think everything modern is bloated and wrong. Be brief and sarcastic.", "color": "#dc2626"},
    {"name": "Excited Intern", "icon": "\U0001f929", "system": "You are an extremely enthusiastic first-year CS student who just discovered transformers. Use lots of exclamation points.", "color": "#2563eb"},
    {"name": "ML Pirate", "icon": "\U0001f3f4\u200d\u2620\ufe0f", "system": "You are a pirate who somehow understands machine learning. Arr.", "color": "#16a34a"},
]

question = "What is a neural network?"
# Use the largest available model
_size_order = {"122B": 0, "35B": 1, "27B": 2, "9B": 3, "8B": 4, "4B": 5, "2B": 6, "1.7B": 7, "0.8B": 8, "0.6B": 9, "0.5B": 10}
target_model = sorted(MODELS, key=lambda m: _size_order.get(m["label"].split("-")[0], 99))[0]
print(f"Using {target_model['label']} on port {target_model['port']}", flush=True)

# Shared state
state = {}
for p in personas:
    state[p["name"]] = {"text": "", "tokens": 0, "elapsed": 0, "tps": 0, "done": False}

def _render_persona_cards():
    cards = ""
    for p in personas:
        s = state[p["name"]]
        text = notebook_helpers.strip_think(s["text"])
        if not text and not s["done"]:
            rendered = "<em>waiting...</em>"
        elif not text and s["done"]:
            rendered = "<em>(empty response)</em>"
        else:
            rendered = notebook_helpers._md(text)
        if s["done"]:
            status = f'{s["tps"]:.1f} tok/s'
        else:
            status = f'streaming... {s["tokens"]} tok'
        cards += f"""
        <div style="flex:1; min-width:250px; background:#f9fafb; border:1px solid #d1d5db;
                    border-left:4px solid {p['color']}; border-radius:0 8px 8px 0; padding:12px 18px;">
          <div style="color:{p['color']}; font-weight:bold; font-size:0.85em; margin-bottom:6px;">
            {p['icon']} {p['name']} · {status}
          </div>
          <div style="color:#1f2937; line-height:1.6; word-wrap:break-word; font-size:0.9em;">
            {rendered}
          </div>
          <div style="color:#9ca3af; font-size:0.75em; margin-top:8px;">
            {s['tokens']} tokens in {s['elapsed']:.1f}s
          </div>
        </div>"""
    return f'<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>'

handle = display(HTML(_render_persona_cards()), display_id=True)

def stream_persona(p):
    t0 = time.time()
    try:
        response = clients[target_model["label"]].chat.completions.create(
            model=target_model["model"],
            messages=[
                {"role": "system", "content": p["system"]},
                {"role": "user", "content": question},
            ],
            max_tokens=256, temperature=PERSONA_TEMP, stream=True,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        token_count = 0
        for chunk in response:
            if chunk.choices and chunk.choices[0].delta.content:
                state[p["name"]]["text"] += chunk.choices[0].delta.content
                token_count += 1
                elapsed = time.time() - t0
                state[p["name"]]["tokens"] = token_count
                state[p["name"]]["elapsed"] = elapsed
                state[p["name"]]["tps"] = token_count / elapsed if elapsed > 0 else 0
    except Exception as e:
        state[p["name"]]["text"] = f"Error: {e}"
    finally:
        elapsed = time.time() - t0
        state[p["name"]]["elapsed"] = elapsed
        if state[p["name"]]["tokens"] > 0:
            state[p["name"]]["tps"] = state[p["name"]]["tokens"] / elapsed
        state[p["name"]]["done"] = True

threads = [threading.Thread(target=stream_persona, args=(p,)) for p in personas]
for t in threads:
    t.start()

while not all(state[p["name"]]["done"] for p in personas):
    time.sleep(0.3)
    handle.update(HTML(_render_persona_cards()))

handle.update(HTML(_render_persona_cards()))
print("Done.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧮 Step 7: Quantization</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> Quantization reduces the precision of model weights (e.g., from 32-bit floats to 4-bit integers). This shrinks the model ~8× with only a small quality loss, making it possible to fit large models in memory.
</div>

**Precision levels and memory cost.** Model weights are normally stored in fp32 (4 bytes per parameter). Quantization compresses them:

| Format | Bytes/param | 7B model size | Notes |
|--------|-------------|---------------|-------|
| fp32 | 4.0 B | 28 GB | Training precision, rarely used for inference |
| bf16 | 2.0 B | 14 GB | Standard inference, full quality |
| INT8 | 1.0 B | 7 GB | Minor quality loss, good speedup |
| 4-bit | 0.5 B | 3.5 GB | Small quality loss for >7B, major size win |
| 2-bit | 0.25 B | 1.75 GB | Noticeable quality degradation |

**How 4-bit quantization works.** Weights are divided into groups of 16–64 values. Within each group, the range [min, max] is found and the 16 levels of a 4-bit integer are mapped linearly across that range. A scale factor (stored in fp16) is saved per group to reconstruct approximate float values at runtime. The overhead of scale factors adds ~5–8% to total size, which is why "4-bit" models are often listed as 4.x bits effective.

**Quality loss is small for large models.** A 70B model quantized to 4-bit typically loses less than 1–2% on standard benchmarks. A 2B model quantized to 4-bit can lose 5–15%. Larger models have more redundancy in their weights, making quantization more forgiving.

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> Quantization is exactly like reducing ADC bit depth. Going from fp32 to 4-bit is analogous to going from a 24-bit ADC to a 4-bit ADC. The key insight is that neural network weight distributions are approximately Gaussian (bell curve) — most weights cluster near zero with few outliers — which is actually the ideal distribution for uniform quantization. Just like a signal that rarely hits full scale can be accurately captured with fewer bits, concentrated weight distributions lose little information when quantized. Section 01c covers the specific formats (GGUF, AWQ, GPTQ, nvfp4) in detail.
</div>

In [12]:
# What quantization actually does to numbers
import mlx.core as mx
from IPython.display import HTML, display

# Real weights are fp32 — 4 bytes each
weights_fp32 = mx.array([0.1234, -0.5678, 0.9012, -0.3456, 0.7890])

# 4-bit quantization: ~16 possible values per weight, loses precision
# Simulate by rounding to nearest 1/8
weights_4bit_approx = mx.round(weights_fp32 * 8) / 8

print("What quantization does to weights:")
print(f"  fp32:   {weights_fp32.tolist()}")
print(f"  ~4-bit: {weights_4bit_approx.tolist()}")
print()

# Size math — this is why quantization matters
params = 32_000_000_000  # 32B parameters

rows = ""
for label, mult, note in [
    ("fp32 (4 bytes/param)", 4, "doesn't fit on most machines"),
    ("bf16 (2 bytes/param)", 2, "still large"),
    ("4-bit (0.5 bytes/param)", 0.5, "fits in 64GB+ Mac"),
]:
    size = params * mult / 1e9
    rows += (
        f"<tr><td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{size:.0f} GB \u2014 {note}</td></tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Precision</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Size (32B params)</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))

What quantization does to weights:
  fp32:   [0.1234000027179718, -0.567799985408783, 0.901199996471405, -0.3456000089645386, 0.7889999747276306]
  ~4-bit: [0.125, -0.625, 0.875, -0.375, 0.75]



Precision,Size (32B params)
fp32 (4 bytes/param),128 GB — doesn't fit on most machines
bf16 (2 bytes/param),64 GB — still large
4-bit (0.5 bytes/param),16 GB — fits in 64GB+ Mac


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💪 Step 8: Stress Test</h2>
</div>

**Why stress test?** Single-prompt benchmarks are misleading — tok/s varies significantly by task complexity, output length, and whether the model is reasoning through a hard problem or pattern-matching a simple one. A proper stress test covers the full behavioral space.

**Three test categories.** The stress test below uses three prompt types that exercise different aspects of model performance:

- **Simple** (e.g., "What is 2+2?"): Tests raw decode speed with minimal reasoning. The model basically pattern-matches and outputs quickly. Measures peak tok/s.
- **Medium** (e.g., technical list): Tests instruction following — the model must structure output, stay on topic, and cover the required items. Moderate compute per token.
- **Creative** (e.g., write a function with docstring): Tests sustained quality over a long generation. The model must maintain coherence across hundreds of tokens. Tok/s may drop slightly as KV cache grows.

**Tok/s varies by task, not just model size.** You'll often see the same model produce 150 tok/s on a trivial prompt and 80 tok/s on a complex coding task. This happens because some model architectures (especially MoE) activate different expert subsets depending on input type — routing overhead and expert load vary.

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> This is like benchmarking a microcontroller with both a tight delay loop (tests raw clock speed) and an interrupt-driven ADC sampling routine (tests real-world throughput including ISR overhead). The loop number looks great on the datasheet; the ADC number tells you what you actually get in your application. Both matter.
</div>

**Concurrent execution via ThreadPoolExecutor.** Each model runs on a separate port, so the tests execute in parallel using Python's `ThreadPoolExecutor`. Each thread sends its request independently, and results are collected as they complete. This mirrors how a production system would serve multiple users simultaneously.

In [ ]:
# Stress test: N tests x N models
import time
import concurrent.futures
from IPython.display import HTML, display

tests = [
    ("Simple", "What is 2+2? Answer with just the number.", 64),
    ("Medium", "List 5 common I2C bus errors and how to debug them.", 512),
    ("Creative", "Write a Python function that reads from a UART buffer and parses NMEA GPS sentences. Include docstring.", 1024),
]

def run_test(m, prompt, max_tokens):
    t0 = time.time()
    r = clients[m["label"]].chat.completions.create(
        model=m["model"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    elapsed = time.time() - t0
    tokens = r.usage.completion_tokens
    return {"label": m["label"], "color": m["color"], "tokens": tokens, "elapsed": elapsed, "tps": tokens / elapsed if elapsed > 0 else 0}

for test_name, prompt, max_tok in tests:
    print(f"Running: {test_name} (max_tokens={max_tok})...", flush=True)
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
        futures = {ex.submit(run_test, m, prompt, max_tok): m["label"] for m in MODELS}
        test_results = []
        for f in concurrent.futures.as_completed(futures):
            test_results.append(f.result())
    # Preserve MODELS order
    label_order = {m["label"]: i for i, m in enumerate(MODELS)}
    test_results.sort(key=lambda r: label_order.get(r["label"], 99))

    rows = ""
    for r in test_results:
        rows += (
            f"<tr>"
            f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['tokens']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['elapsed']:.1f}s</td>"
            f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['tps']:.1f}</td>"
            f"</tr>"
        )
    display(HTML(f"""
    <div style="color:#1e3a5f; font-weight:bold; margin:8px 0 4px 0;">{test_name}</div>
    <table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:4px 0 12px 0; font-family:monospace; min-width:400px;">
      <thead><tr style="background:#1e3a5f;">
        <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Tokens</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Time</th>
        <th style="padding:8px 12px; color:white; text-align:left;">tok/s</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>
    """))
    print(f"  ✓ {test_name} complete\n", flush=True)

print("All stress tests done.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🗂️ Step 9: KV Cache — Why Second Turns Are Faster</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> The KV (key-value) cache stores attention matrices from prior tokens so the model doesn’t recompute them on every turn. This speeds up multi-turn conversations but grows with context length — long conversations eat more RAM.
</div>

**Why attention needs prior tokens.** The transformer attention mechanism computes a weighted sum over all previous token representations. For every new token generated, the model needs the Key (K) and Value (V) matrices for *every token that came before it* in the context. Without a cache, this means recomputing all prior K/V pairs on every decode step — O(n²) total work for a sequence of length n.

**With KV cache: O(n) amortized.** The KV cache stores the K and V tensors from all prior tokens. On each new decode step, only the *new* token's K/V pair is computed and appended to the cache. Total work becomes O(n) — linear in context length. This is the single biggest optimization that makes autoregressive generation practical.

**Two phases of inference.**
1. **Prefill** — The prompt is processed in a single batched forward pass. All K/V pairs are computed simultaneously and stored. This phase determines Time-to-First-Token (TTFT). Long prompts → higher TTFT.
2. **Decode** — One token is generated per step. Each step appends one K/V pair to the cache. This phase determines tok/s. The bottleneck is memory bandwidth (streaming the full cache + weights per step).

**Multi-turn conversations reuse the cache.** When you send a follow-up message, the server can reuse the KV cache from the prior turns — only the new message tokens need to be prefilled. This is why the second turn in a conversation often has lower TTFT than the first.

<div style="border-left:4px solid #f59e0b; background:#fffbeb; padding:10px 16px; border-radius:0 6px 6px 0; margin:12px 0; font-size:0.9em;">
  <b>🔌 Embedded analogy:</b> The KV cache is like a DMA descriptor ring buffer. Instead of reconfiguring DMA from scratch for every transfer (recomputing all K/V from scratch), you maintain a ring of pre-configured descriptors (cached K/V pairs) and only add new descriptors for new data. The ring grows as context length grows — just as your ring buffer fills up when data arrives faster than it's consumed.
</div>

**Memory cost grows linearly with context.** KV cache size = `2 × num_layers × num_heads × head_dim × context_length × bytes_per_element`. For a 32-layer, 32-head, 128-dim model in bf16 at 32K context: ~16 GB. This is why very long contexts require significant RAM headroom beyond the model itself.

**Hybrid models reduce KV cache pressure.** Some architectures (covered in Section 01b) use Grouped Query Attention (GQA) for only a fraction of their layers — for example, the DeltaNet/GQA hybrid uses GQA in ~25% of layers. GQA shares K/V heads across multiple query heads, reducing KV cache size by 4–8× for those layers.

In [152]:
# KV cache demo: measure time-to-first-token (TTFT) for cold vs cached
# TTFT isolates prompt processing time — the phase KV cache speeds up
import time
from IPython.display import HTML, display

def kv_cache_test(m):
    """Measure TTFT for cold (new conversation) vs cached (follow-up turn)."""
    # Turn 1 — cold: no prior context, must process prompt from scratch
    messages_1 = [{"role": "user", "content": "I'm going to ask you 3 questions about embedded systems. Ready?"}]
    t0 = time.time()
    stream_1 = clients[m["label"]].chat.completions.create(
        model=m["model"], messages=messages_1, max_tokens=50, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    ttft_cold = None
    reply_1 = ""
    for chunk in stream_1:
        if ttft_cold is None and chunk.choices and chunk.choices[0].delta.content:
            ttft_cold = time.time() - t0
        if chunk.choices and chunk.choices[0].delta.content:
            reply_1 += chunk.choices[0].delta.content
    if ttft_cold is None:
        ttft_cold = time.time() - t0

    # Turn 2 — cached: server reuses KV cache from turn 1
    messages_2 = messages_1 + [
        {"role": "assistant", "content": reply_1},
        {"role": "user", "content": "Question 1: What's the difference between I2C and SPI?"},
    ]
    t1 = time.time()
    stream_2 = clients[m["label"]].chat.completions.create(
        model=m["model"], messages=messages_2, max_tokens=50, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    ttft_cached = None
    for chunk in stream_2:
        if ttft_cached is None and chunk.choices and chunk.choices[0].delta.content:
            ttft_cached = time.time() - t1
            break  # got what we need
    if ttft_cached is None:
        ttft_cached = time.time() - t1

    speedup = ttft_cold / ttft_cached if ttft_cached > 0 else 0
    return {"label": m["label"], "color": m["color"], "cold": ttft_cold, "cached": ttft_cached, "speedup": speedup}

print("Measuring time-to-first-token (TTFT)...", flush=True)
results = []
for m in MODELS:
    print(f"  Testing {m['label']}...", flush=True)
    results.append(kv_cache_test(m))

rows = ""
for r in results:
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['cold']*1000:.0f} ms</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['cached']*1000:.0f} ms</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['speedup']:.1f}x</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">TTFT cold</th>
    <th style="padding:8px 12px; color:white; text-align:left;">TTFT cached</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Speedup</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  TTFT = time-to-first-token — measures prompt processing, not generation.
  Cached turn reuses KV matrices from turn 1, so only new tokens need processing.
</div>
"""))
print("Done.")

Measuring time-to-first-token (TTFT)...
  Testing 122B...
  Testing 35B...
  Testing 2B...


Model,TTFT cold,TTFT cached,Speedup
122B,996 ms,321 ms,3.1x
35B,433 ms,285 ms,1.5x
2B,196 ms,175 ms,1.1x


Done.


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

- **Unified memory** lets Apple Silicon run models that don't fit on discrete GPUs
- **MLX** serves models locally via an OpenAI-compatible API
- **4-bit quantization** shrinks models 8× with minimal quality loss
- **Temperature** controls randomness: 0 = deterministic, 1.5 = chaotic
- **System prompts** steer behavior without changing model weights
- **tok/s** is bounded by memory bandwidth, not compute
- **KV cache** speeds up multi-turn conversations by reusing prior computations
- **Running multiple models simultaneously** reveals the quality/speed tradeoff viscerally
- **Model size vs quality:** sub-1B is instant but shallow, mid-range (4-9B) hits the sweet spot, 35B+ delivers near-frontier quality
- Everything here ran locally — no data left your machine

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What’s Next</h2>
</div>

- **Section 01b — Model Datasheet:** Architecture deep dive — DeltaNet/GQA hybrid attention, MoE routing (which experts fire and why), and the KV cache memory math that determines how long your context can be.
- **Section 01c — Inference Optimization:** Bandwidth analysis (why memory bandwidth — not FLOPS — is the real bottleneck), speculative decoding, prefix caching, and the quantization format zoo (GGUF, AWQ, GPTQ, nvfp4).
- **Section 02 — Tokenization and Embeddings:** How text becomes numbers — BPE tokenization, token count analysis, and a tour of embedding spaces that explains why "king − man + woman ≈ queen" actually works.
- **Section 03 — Prompting Techniques:** Zero-shot, few-shot, and chain-of-thought prompting — with benchmarks showing when each technique wins and when it wastes tokens.